# Multimodal Topic Mining and Image Classification
## Advanced Customer Analytics - Visual Data Predictions Assignment

### Dataset Declaration
**Dataset:** MS COCO 2014 Train Dataset
- **Source:** Microsoft COCO (Common Objects in Context)
- **Size:** 82,783 images with 5 captions each (using subset of 5,000+ for this assignment)
- **URL:** https://cocodataset.org/
- **Description:** Diverse images containing everyday scenes with common objects in their natural context
- **Domains:** People, animals, vehicles, furniture, sports, food, indoor/outdoor scenes
- **Why this dataset:** COCO provides high-quality, diverse images across multiple domains with detailed captions, making it ideal for multimodal topic modeling

In [ ]:
# # Install required packages
# !pip install -q pokemontcgsdk
# !pip install -q bertopic
# !pip install -q scikit-learn
# !pip install -q pillow
# !pip install -q pandas
# !pip install -q matplotlib
# !pip install -q seaborn
# !pip install -q requests
# !pip install -q tqdm
!pip install datasets

In [12]:
import os
import requests
import numpy as np
import pandas as pd
from PIL import Image
from io import BytesIO
from tqdm import tqdm
from collections import defaultdict

# Pokemon TCG SDK
from pokemontcgsdk import Card, Set, Type, Supertype, Subtype, Rarity
from pokemontcgsdk import RestClient

# BERTopic
from bertopic import BERTopic
from bertopic.backend import MultiModalBackend
from bertopic.representation import VisualRepresentation
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer

# ML models
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

/opt/homebrew/Caskroom/miniconda/base/envs/visual_data_predictions/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [24]:
keyFile = open('./api-key.txt', 'r')

if keyFile is None:
    print("API key file not found. Rate limit available will be: 20,000 requests/day.")
else:
    api_key = keyFile.readline().rstrip()
    RestClient.configure(api_key)
    print("API key found. Rate limit available will be: 100,000 requests/day.")

API key found. Rate limit available will be: 100,000 requests/day.


In [30]:
def download_from_huggingface(num_cards=1000, save_dir='pokemon_cards'):
    """Download from pre-made dataset - NO API NEEDED!"""
    from datasets import load_dataset
    
    os.makedirs(f"{save_dir}/images", exist_ok=True)
    
    print("Loading dataset from Hugging Face...")
    dataset = load_dataset("TheFusion21/PokemonCards", split="train")
    dataset = dataset.select(range(min(num_cards, len(dataset))))
    
    image_paths = []
    captions = []
    cards_data = []
    
    print(f"Downloading {len(dataset)} cards...")
    
    for idx, item in enumerate(tqdm(dataset, desc="Processing cards")):
        try:
            # Download image - field is 'image_url' not 'image'!
            img_url = item['image_url']
            
            if not img_url:
                continue
                
            response = requests.get(img_url, timeout=10)
            response.raise_for_status()
            
            img = Image.open(BytesIO(response.content)).convert('RGB')
            
            # Save image
            img_path = f"{save_dir}/images/card_{idx:05d}.jpg"
            img.save(img_path, 'JPEG', quality=95)
            
            # Get pre-made caption
            caption = item.get('caption', item.get('name', 'Pokemon card'))
            
            # Store data
            image_paths.append(img_path)
            captions.append(caption)
            cards_data.append({
                'id': item.get('id', f'card_{idx}'),
                'name': item.get('name', 'Unknown'),
                'hp': item.get('hp', 'Unknown'),
                'set_name': item.get('set_name', 'Unknown'),
                'image_path': img_path,
                'caption': caption
            })
            
        except Exception as e:
            # Print errors so we can debug
            if idx < 5:  # Print first few errors
                print(f"\nError on card {idx}: {e}")
            continue
    
    # Save metadata
    df = pd.DataFrame(cards_data)
    df.to_csv(f'{save_dir}/pokemon_cards_metadata.csv', index=False)
    
    print(f"\n✓ Downloaded {len(image_paths)} cards successfully!")
    return image_paths, captions, cards_data

In [31]:
# Download the cards using Hugging Face!
print("Starting Pokemon card download from Hugging Face...")
print()

image_paths, captions, cards_metadata = download_from_huggingface(num_cards=1000)

# Display some examples
print("\n" + "=" * 70)
print("SAMPLE CAPTIONS")
print("=" * 70)
for i in range(min(5, len(captions))):
    print(f"\n{i+1}. {captions[i][:200]}...")

Starting Pokemon card download from Hugging Face...

Loading dataset from Hugging Face...


Processing cards: 100%|██████████| 1000/1000 [05:50<00:00,  2.85it/s]


✓ Downloaded 999 cards successfully!

SAMPLE CAPTIONS

1. A Basic, SP Pokemon Card of type Darkness with the title Absol G and 70 HP of rarity Rare Holo from the set Supreme Victors.  It has the attack Feint Attack with the cost Darkness, the energy cost 1 w...

2. A Stage 1 Pokemon Card of type Colorless with the title Aerodactyl and 70 HP of rarity Rare Holo evolved from Mysterious Fossil from the set Legend Maker.  It has the attack Power Blow with the cost C...

3. A Basic Pokemon Card of type Grass with the title Weedle and 50 HP of rarity Common from the set Primal Clash and the flavor text: Its poison stinger is very powerful. Its bright-colored body is inten...

4. A Basic Pokemon Card of type Grass with the title Caterpie and 50 HP of rarity None from the set McDonald's Collection 2019.  It has the attack Surprise Attack with the cost Grass, the energy cost 1 a...

5. A Stage 1 Pokemon Card of type Water with the title Azumarill and 80 HP of rarity Rare Holo evolved from Mari